# Define proxy-state labels

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path("/workspace")
INPUT_DIR = PROJECT_ROOT / "output" / "09_layer2_dataset"
OUTPUT_DIR = PROJECT_ROOT / "output" / "11_layer2_proxy_labels"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = INPUT_DIR / "layer2_master_table.csv.gz"

assert MASTER_PATH.exists(), f"Missing input master table: {MASTER_PATH}"
print("MASTER_PATH:", MASTER_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
df = pd.read_csv(MASTER_PATH)
print("Shape:", df.shape)
display(df.head(3))

In [ ]:
required_cols = ["_record_key", "clip_id"]
missing_required = [c for c in required_cols if c not in df.columns]
assert not missing_required, f"Missing required columns: {missing_required}"

def pick_col(candidates, frame):
    for c in candidates:
        if c in frame.columns:
            return c
    return None

MAIN_COL = pick_col(
    ["main_category_final", "main_category", "MainCategory", "main_category_label"],
    df
)
SUB_COL = pick_col(
    ["sub_category_final", "subcategory_final", "sub_category", "SubCategory", "sub_category_label"],
    df
)
FARM_COL = pick_col(["farm", "Farm", "farm_id"], df)
ZONE_COL = pick_col(["barn_zone", "zone", "BarnZone"], df)
MIC_COL = pick_col(["microphone", "mic", "Microphone"], df)
PLACEMENT_COL = pick_col(["mic_placement_context", "placement_context", "mic_context"], df)

assert MAIN_COL is not None, "Could not find main category column."
assert SUB_COL is not None, "Could not find subcategory column."

print("Resolved columns:")
print("  MAIN_COL:", MAIN_COL)
print("  SUB_COL:", SUB_COL)
print("  FARM_COL:", FARM_COL)
print("  ZONE_COL:", ZONE_COL)
print("  MIC_COL:", MIC_COL)
print("  PLACEMENT_COL:", PLACEMENT_COL)

In [ ]:
work = df.copy()

for col in [MAIN_COL, SUB_COL, FARM_COL, ZONE_COL, MIC_COL, PLACEMENT_COL]:
    if col is not None:
        work[col] = work[col].fillna("-").astype(str).str.strip()

work["main_label_raw"] = work[MAIN_COL]
work["sub_label_raw"] = work[SUB_COL]

display(work[["main_label_raw", "sub_label_raw"]].drop_duplicates().sort_values(["main_label_raw", "sub_label_raw"]).head(20))

In [ ]:
preserved_map = {
    "Non-Vocal Sounds": "physiological_nonvocal",
    "Feeding & Hunger Related": "feeding_hunger",
    "Estrus & Mating Behavior": "reproductive_estrus",
    "Social Recognition & Interaction": "social_interaction",
    "Distress & Pain": "distress_pain",
    "Water & Thirst Related": "water_thirst",
    "Milking & Handling": "milking_handling",
    "Maternal & Calf Communication": "maternal_calf",
    "Environmental & Situational": "environmental_situational",
}

functional_map = {
    "Non-Vocal Sounds": "physiological_nonvocal",
    "Feeding & Hunger Related": "ingestion_resource",
    "Water & Thirst Related": "ingestion_resource",
    "Estrus & Mating Behavior": "reproductive_estrus",
    "Social Recognition & Interaction": "social_contact",
    "Maternal & Calf Communication": "social_contact",
    "Distress & Pain": "stress_discomfort",
    "Milking & Handling": "stress_discomfort",
    "Environmental & Situational": "stress_discomfort",
}

def infer_proxy_subfamily(sub_label: str) -> str:
    s = (sub_label or "").strip()
    if s in {"Breathing_Respiratory_Sounds", "Chewing_Rumination_Sounds", "Cough_Calls", "Licking_Sounds", "Movement_Associated_Sounds"}:
        return "physiological_nonvocal_subfamily"
    if "Feed" in s or "Hunger" in s:
        return "feeding_hunger_subfamily"
    if "Water" in s or "Drinking" in s or "Thirst" in s or "Dehydration" in s:
        return "water_thirst_subfamily"
    if "Estrus" in s or "Mating" in s:
        return "reproductive_subfamily"
    if "Mother" in s or "Calf" in s or "Maternal" in s:
        return "maternal_subfamily"
    if "Social" in s or "Greeting" in s or "Recognition" in s or "Coordination" in s or "Contact" in s or "Response_Exchange" in s or "Proximity" in s:
        return "social_subfamily"
    if "Pain" in s or "Distress" in s or "Discomfort" in s or "Stress" in s or "Restraint" in s or "Injury" in s or "Illness" in s:
        return "stress_discomfort_subfamily"
    if "Milking" in s or "Handling" in s:
        return "handling_subfamily"
    if "Novel_Environment" in s:
        return "environmental_subfamily"
    return "unmapped_subfamily"

work["proxy_main_preserved"] = work["main_label_raw"].map(preserved_map).fillna("unmapped_proxy")
work["proxy_functional_v1"] = work["main_label_raw"].map(functional_map).fillna("unmapped_proxy")
work["proxy_subfamily_hint"] = work["sub_label_raw"].map(infer_proxy_subfamily)

work["is_nonvocal"] = (work["proxy_functional_v1"] == "physiological_nonvocal").astype(int)
work["is_distress_related"] = (work["proxy_functional_v1"] == "stress_discomfort").astype(int)
work["is_resource_related"] = (work["proxy_functional_v1"] == "ingestion_resource").astype(int)

In [ ]:
def encode_labels(series: pd.Series) -> tuple[pd.Series, dict]:
    labels = sorted(series.dropna().astype(str).unique().tolist())
    mapping = {label: idx for idx, label in enumerate(labels)}
    return series.map(mapping), mapping

for label_col in ["proxy_main_preserved", "proxy_functional_v1", "proxy_subfamily_hint"]:
    work[f"{label_col}_id"], mapping = encode_labels(work[label_col])
    print(label_col, mapping)

In [ ]:
dist_main = work["proxy_main_preserved"].value_counts(dropna=False).sort_values(ascending=False)
dist_func = work["proxy_functional_v1"].value_counts(dropna=False).sort_values(ascending=False)
dist_subfamily = work["proxy_subfamily_hint"].value_counts(dropna=False).sort_values(ascending=False)

print("proxy_main_preserved:")
display(dist_main.to_frame("count"))

print("proxy_functional_v1:")
display(dist_func.to_frame("count"))

print("proxy_subfamily_hint:")
display(dist_subfamily.to_frame("count"))

In [ ]:
farm_crosstabs = {}
if FARM_COL is not None:
    ct_preserved = pd.crosstab(work[FARM_COL], work["proxy_main_preserved"], normalize="index")
    ct_func = pd.crosstab(work[FARM_COL], work["proxy_functional_v1"], normalize="index")
    farm_crosstabs["proxy_main_preserved"] = ct_preserved
    farm_crosstabs["proxy_functional_v1"] = ct_func

    print("Per-farm distribution: proxy_main_preserved")
    display(ct_preserved)

    print("Per-farm distribution: proxy_functional_v1")
    display(ct_func)
else:
    print("No farm column found; skipping per-farm cross-tabs.")

In [ ]:
mapping_rows = []
for main_label in sorted(work["main_label_raw"].unique().tolist()):
    subset = work.loc[work["main_label_raw"] == main_label, "sub_label_raw"].dropna().astype(str).unique().tolist()
    mapping_rows.append({
        "main_label_raw": main_label,
        "proxy_main_preserved": preserved_map.get(main_label, "unmapped_proxy"),
        "proxy_functional_v1": functional_map.get(main_label, "unmapped_proxy"),
        "n_subcategories_seen": len(subset),
        "subcategories_seen": " | ".join(sorted(subset)),
    })

mapping_df = pd.DataFrame(mapping_rows).sort_values("main_label_raw").reset_index(drop=True)
display(mapping_df)

In [ ]:
label_cols = [
    "_record_key", "clip_id", "main_label_raw", "sub_label_raw",
    "proxy_main_preserved", "proxy_main_preserved_id",
    "proxy_functional_v1", "proxy_functional_v1_id",
    "proxy_subfamily_hint", "proxy_subfamily_hint_id",
    "is_nonvocal", "is_distress_related", "is_resource_related",
]
for extra in [FARM_COL, ZONE_COL, MIC_COL, PLACEMENT_COL]:
    if extra is not None and extra not in label_cols:
        label_cols.append(extra)

proxy_labels_df = work[label_cols].copy()

proxy_labels_path = OUTPUT_DIR / "proxy_labels_master.csv.gz"
mapping_path = OUTPUT_DIR / "proxy_label_mapping_table.csv"
proxy_labels_df.to_csv(proxy_labels_path, index=False, compression="gzip")
mapping_df.to_csv(mapping_path, index=False)

for target in ["proxy_main_preserved", "proxy_functional_v1", "proxy_subfamily_hint"]:
    target_df = proxy_labels_df[["_record_key", "clip_id", target, f"{target}_id"]].copy()
    target_df.to_csv(OUTPUT_DIR / f"{target}.csv.gz", index=False, compression="gzip")

print("Saved:")
print(" ", proxy_labels_path)
print(" ", mapping_path)

In [ ]:
diag = {
    "build_ts_utc": datetime.now(timezone.utc).isoformat(),
    "input_master_path": str(MASTER_PATH),
    "output_dir": str(OUTPUT_DIR),
    "n_clips": int(len(work)),
    "main_label_col": MAIN_COL,
    "sub_label_col": SUB_COL,
    "farm_col": FARM_COL,
    "proxy_main_preserved_counts": dist_main.to_dict(),
    "proxy_functional_v1_counts": dist_func.to_dict(),
    "proxy_subfamily_hint_counts": dist_subfamily.to_dict(),
    "binary_targets": {
        "is_nonvocal_positive": int(work["is_nonvocal"].sum()),
        "is_distress_related_positive": int(work["is_distress_related"].sum()),
        "is_resource_related_positive": int(work["is_resource_related"].sum()),
    },
    "unmapped": {
        "proxy_main_preserved": int((work["proxy_main_preserved"] == "unmapped_proxy").sum()),
        "proxy_functional_v1": int((work["proxy_functional_v1"] == "unmapped_proxy").sum()),
        "proxy_subfamily_hint": int((work["proxy_subfamily_hint"] == "unmapped_subfamily").sum()),
    }
}
if FARM_COL is not None:
    diag["per_farm_counts"] = work[FARM_COL].value_counts(dropna=False).to_dict()

diag_path = OUTPUT_DIR / "proxy_label_diagnostics.json"
with open(diag_path, "w", encoding="utf-8") as f:
    json.dump(diag, f, indent=2)

print("Diagnostics saved:", diag_path)
diag